# 03 — Gold metrics (OSS)

Build a compact Gold summary from Silver tenant-metrics Delta (local filesystem).


In [ ]:
from pathlib import Path
from ambient_pipeline.notebook_bootstrap import ensure_pipeline_on_path, apply_spark_tuning

ROOT = ensure_pipeline_on_path(Path.cwd())
assert ROOT is not None, "Run from ambient-core checkout (lib/ambient_pipeline missing)"
print(f"repo root: {ROOT}")


In [ ]:
from ambient_pipeline.perf import create_local_spark

spark = create_local_spark(app_name="ambient-oss-notebooks", shuffle_partitions=4)
apply_spark_tuning(spark)
print(spark.version)


In [ ]:
from pyspark.sql import functions as F
from ambient_pipeline.storage_paths import resolve_table_path

out_base = str(ROOT / ".lakehouse" / "demo")
silver_table = resolve_table_path("local", out_base, "demo", "silver", "tenant_metrics")
gold_table = resolve_table_path("local", out_base, "demo", "gold", "metric_summary")

silver = spark.read.format("delta").load(silver_table)
gold = (
    silver.groupBy("_bronze_org_id", "industry")
    .agg(
        F.count("*").alias("metric_rows"),
        F.countDistinct("metric_id").alias("distinct_metrics"),
        F.countDistinct("name").alias("distinct_names"),
    )
)
(
    gold.write.format("delta").mode("overwrite").save(gold_table)
)
print(f"gold rows={spark.read.format('delta').load(gold_table).count()} path={gold_table}")
gold.show(truncate=False)
